# Making SIMSOPT GPU native: extended convergence and visualization

Select **Runtime > Change runtime type > GPU**, then run all cells. This experiment extends the scaled full-engineering comparison to 300 iterations. Backend parity is evaluated over the first 25 accepted iterates; final acceptance requires stationary solutions and equivalent normalized-field and constraint quality. CPU and GPU final surfaces (`.vts`) and coils (`.vtu`) are exported for ParaView inspection. The artifact is always downloaded, including when a gate fails.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo = Path("/content/simsopt")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "gpu-native-objective", "https://github.com/PedroFranciscoGil/simsopt.git", str(repo)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "switch", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "pytest", "pyevtk"], cwd=repo, check=True)
os.chdir(repo)
source_root = repo / "src"
sys.path.insert(0, str(source_root))
for module_name in tuple(sys.modules):
    if module_name == "simsopt" or module_name.startswith("simsopt."):
        del sys.modules[module_name]
importlib.invalidate_caches()
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(revision)

In [ ]:
import jax
import simsopt
from simsopt.gpu import backend_report

resolved_package = Path(simsopt.__file__).resolve()
print(f"Imported SIMSOPT from {resolved_package}")
assert source_root in resolved_package.parents, resolved_package
report = backend_report()
print(report)
assert jax.default_backend() == "gpu", report

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/gpu"], check=True)

In [ ]:
import shutil

artifact_root = Path("/content/simsopt-extended-convergence")
if artifact_root.exists():
    shutil.rmtree(artifact_root)
artifact_root.mkdir()
result_file = artifact_root / "stress-extended-convergence.json"
env = os.environ.copy()
env["OMP_NUM_THREADS"] = "1"
env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
subprocess.run([sys.executable, "benchmarks/gpu/compare_scipy_trajectories.py", "--problem", "stress", "--maxiter", "300", "--trajectory-parity-iterations", "25", "--current-scale", "100000", "--target-tile-size", "1024", "--source-tile-size", "4320", "--visualization-dir", str(artifact_root), "--output", str(result_file)], cwd=repo, env=env, check=True)

In [ ]:
import json

result = json.loads(result_file.read_text())
failed = {name: gate for name, gate in result["gates"].items() if not gate["passed"]}
summary = {"all_gates_passed": result["all_gates_passed"], "solver": result["solver"], "comparison": result["comparison"], "cpu_final_metrics": result["cpu"]["final_metrics"], "gpu_final_metrics": result["gpu"]["final_metrics_from_cpu_oracle"], "visualization": result["visualization"], "failed_gates": failed}
print(json.dumps(summary, indent=2))
assert result["schema_version"] == 3
assert result["solver"]["trajectory_parity_iterations"] == 25
assert result["coordinate_scaling"]["current_scale_amperes"] == 100000.0
expected_files = ["cpu_final_surface.vts", "cpu_final_coils.vtu", "gpu_final_surface.vts", "gpu_final_coils.vtu"]
for filename in expected_files:
    path = artifact_root / filename
    assert path.is_file() and path.stat().st_size > 0, path

## ParaView

Open either `cpu_final_surface.vts` or `gpu_final_surface.vts`, select **Surface**, and color by `B_dot_n_over_abs_B` (signed) or `abs_B_dot_n_over_abs_B`. Add the matching `*_coils.vtu` file to inspect the optimized physical coil set around the surface.

In [ ]:
from google.colab import files

archive = shutil.make_archive("/content/simsopt-extended-convergence", "zip", artifact_root)
files.download(archive)